# NB07 — Model Evaluation & Comparison

> **Pipeline position:** NB04 (YOLO11n) → NB05 (YOLO11s) → NB06 (RT-DETR, deferred) → **NB07 (Evaluation)** → NB08 (Deployment)

## What this notebook does

| Section | Purpose |
|---|---|
| 0. Setup | Load results CSVs, model weights, paths |
| 1. Model Overview | Architecture comparison (params, size, FLOPs) |
| 2. Overall Performance | mAP50 / mAP50-95 bar charts (val + test) |
| 3. Per-Class AP50 | Grouped bar chart — all 5 classes on test set |
| 4. Training Dynamics | Overlaid training curves (losses, mAP50) |
| 5. Minority Class Deep Dive | vest / no-vest AP50 progression across checkpoints |
| 6. Confusion Matrices | Side-by-side normalized confusion matrices |
| 7. Radar Plot | Multi-axis per-class comparison |
| 8. Speed vs Accuracy | Inference latency comparison + tradeoff plot |
| 9. Final Summary | Recommendation table + deployment guidance |

## Models compared

| Model | Status | Test mAP50 |
|---|---|---|
| YOLO11n | Trained (NB04) | 88.6% |
| YOLO11s | Trained (NB05) | 93.2% |
| RT-DETR-l | Deferred (NB06) | — |

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image as IPImage, Markdown

import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# ---- Paths ----
PROJECT_ROOT = Path('../')
DATA_YAML    = PROJECT_ROOT / 'data' / 'ppe_dataset.yaml'
CFG_JSON     = PROJECT_ROOT / 'data' / 'training_config.json'
RESULTS_DIR  = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES  = ['hardhat', 'no-hardhat', 'vest', 'no-vest', 'person']
MINORITY_CLS = ['vest', 'no-vest']

# ---- Auto-detect RUNS_DIR ----
_candidate_dirs = [
    Path('runs/detect'),
    PROJECT_ROOT / 'runs' / 'detect',
]
RUNS_DIR = None
for _d in _candidate_dirs:
    if _d.is_dir() and any(_d.iterdir()):
        RUNS_DIR = _d
        break
assert RUNS_DIR is not None, 'Could not find runs/detect with training results'

# ---- Load configs ----
with open(CFG_JSON) as f:
    ALL_CONFIGS = json.load(f)

# ---- Auto-detect run folders ----
def find_best_run(prefix):
    """Find the latest run dir matching prefix that has best.pt."""
    candidates = []
    for runs_dir in [Path('runs/detect'), PROJECT_ROOT / 'runs' / 'detect']:
        if not runs_dir.is_dir():
            continue
        for p in runs_dir.iterdir():
            if p.name.startswith(prefix) and p.is_dir():
                candidates.append(p)
    with_best = [p for p in candidates if (p / 'weights' / 'best.pt').exists()]
    pool = with_best if with_best else candidates
    if not pool:
        return None
    return max(pool, key=lambda p: p.stat().st_mtime)

YOLO11N_DIR = find_best_run('ppe_yolo11n_1280')
YOLO11S_DIR = find_best_run('ppe_yolo11s_1280')

assert YOLO11N_DIR is not None, 'YOLO11n run not found'
assert YOLO11S_DIR is not None, 'YOLO11s run not found'

# ---- Load comparison CSV ----
NB07_CSV = RESULTS_DIR / 'nb07_model_comparison.csv'
assert NB07_CSV.exists(), f'Comparison CSV not found: {NB07_CSV}'
comp_df = pd.read_csv(NB07_CSV)

# ---- Load checkpoint eval CSVs ----
nb04_ckpt_df = pd.read_csv(RESULTS_DIR / 'nb04_checkpoint_evals.csv')
nb05_ckpt_df = pd.read_csv(RESULTS_DIR / 'nb05_checkpoint_evals.csv')

# ---- Load training curves ----
yolo11n_results = pd.read_csv(YOLO11N_DIR / 'results.csv')
yolo11n_results.columns = yolo11n_results.columns.str.strip()
yolo11s_results = pd.read_csv(YOLO11S_DIR / 'results.csv')
yolo11s_results.columns = yolo11s_results.columns.str.strip()

# ---- Convenience row accessors ----
n_row = comp_df[comp_df['model'].str.contains('yolo11n')].iloc[0]
s_row = comp_df[comp_df['model'].str.contains('yolo11s')].iloc[0]

print(f'YOLO11n run : {YOLO11N_DIR.resolve()}')
print(f'YOLO11s run : {YOLO11S_DIR.resolve()}')
print(f'Comparison  : {NB07_CSV}')
print(f'\nModels in comparison table:')
for _, row in comp_df.iterrows():
    print(f"  {row['model']}: val_mAP50={row['val_map50']:.4f}, test_mAP50={row['test_map50']:.4f}")

---
## 1. Model Architecture Overview

Compare model complexity, size, and theoretical compute.

In [ ]:
# ============================================================
# 1. MODEL ARCHITECTURE OVERVIEW
# Load each best.pt to extract model info.
# ============================================================
model_info = []

for label, run_dir, cfg_key in [
    ('YOLO11n', YOLO11N_DIR, 'yolo11n'),
    ('YOLO11s', YOLO11S_DIR, 'yolo11s'),
]:
    weights = run_dir / 'weights' / 'best.pt'
    model = YOLO(str(weights))

    # Extract model stats
    n_params = sum(p.numel() for p in model.model.parameters())
    weight_size_mb = weights.stat().st_size / 1e6

    # GFLOPs from model info
    info = model.info(verbose=False)

    cfg = ALL_CONFIGS[cfg_key]

    model_info.append({
        'Model': label,
        'Parameters': f'{n_params/1e6:.1f}M',
        'Params (raw)': n_params,
        'Weight Size (MB)': f'{weight_size_mb:.1f}',
        'Training Batch': cfg['batch'],
        'Effective Batch': cfg['nbs'],
        'Image Size': cfg['imgsz'],
    })

    del model
    torch.cuda.empty_cache()

info_df = pd.DataFrame(model_info)
display_cols = ['Model', 'Parameters', 'Weight Size (MB)', 'Training Batch',
                'Effective Batch', 'Image Size']
print('Model Architecture Comparison:')
print(info_df[display_cols].to_string(index=False))
print(f'\nYOLO11s has {model_info[1]["Params (raw)"] / model_info[0]["Params (raw)"]:.1f}x more parameters than YOLO11n')

---
## 2. Overall Performance Comparison (val + test)

In [ ]:
# ============================================================
# 2. OVERALL PERFORMANCE — mAP50 and mAP50-95 bar charts
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Overall Performance Comparison', fontsize=14, fontweight='bold')

models = comp_df['model'].apply(
    lambda x: 'YOLO11n' if 'yolo11n' in x else 'YOLO11s'
).values
colors = ['#4C72B0', '#DD8452']
x = np.arange(len(models))
bar_w = 0.35

# mAP50
ax = axes[0]
val_bars = ax.bar(x - bar_w/2, comp_df['val_map50'] * 100, bar_w,
                  label='Val', color=colors[0], alpha=0.85)
test_bars = ax.bar(x + bar_w/2, comp_df['test_map50'] * 100, bar_w,
                   label='Test', color=colors[1], alpha=0.85)
ax.set_title('mAP50', fontweight='bold')
ax.set_ylabel('mAP50 (%)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(80, 100)
ax.legend()
for bars in [val_bars, test_bars]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.1f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold')

# mAP50-95
ax = axes[1]
val_bars = ax.bar(x - bar_w/2, comp_df['val_map5095'] * 100, bar_w,
                  label='Val', color=colors[0], alpha=0.85)
test_bars = ax.bar(x + bar_w/2, comp_df['test_map5095'] * 100, bar_w,
                   label='Test', color=colors[1], alpha=0.85)
ax.set_title('mAP50-95', fontweight='bold')
ax.set_ylabel('mAP50-95 (%)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(50, 75)
ax.legend()
for bars in [val_bars, test_bars]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.1f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_overall_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

# Delta summary
print(f'\nYOLO11s vs YOLO11n improvements:')
print(f'  Val  mAP50   : {(s_row["val_map50"] - n_row["val_map50"])*100:+.1f}%')
print(f'  Test mAP50   : {(s_row["test_map50"] - n_row["test_map50"])*100:+.1f}%')
print(f'  Val  mAP50-95: {(s_row["val_map5095"] - n_row["val_map5095"])*100:+.1f}%')
print(f'  Test mAP50-95: {(s_row["test_map5095"] - n_row["test_map5095"])*100:+.1f}%')

---
## 3. Per-Class AP50 Comparison (Test Set)

In [ ]:
# ============================================================
# 3. PER-CLASS AP50 — grouped bar chart (test set)
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(CLASS_NAMES))
bar_w = 0.35

yolo11n_ap50 = [n_row[f'test_ap50_{cls}'] * 100 for cls in CLASS_NAMES]
yolo11s_ap50 = [s_row[f'test_ap50_{cls}'] * 100 for cls in CLASS_NAMES]

bars1 = ax.bar(x - bar_w/2, yolo11n_ap50, bar_w, label='YOLO11n',
               color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + bar_w/2, yolo11s_ap50, bar_w, label='YOLO11s',
               color='#DD8452', alpha=0.85)

ax.set_title('Per-Class AP50 on Test Set', fontsize=14, fontweight='bold')
ax.set_ylabel('AP50 (%)')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylim(75, 100)
ax.legend(fontsize=11)

# Annotate bars
for bars in [bars1, bars2]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.1f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=9, fontweight='bold')

# Highlight minority classes
for i, cls in enumerate(CLASS_NAMES):
    if cls in MINORITY_CLS:
        ax.axvspan(i - 0.45, i + 0.45, alpha=0.08, color='red')
        ax.text(i, 76, 'MINORITY', ha='center', fontsize=8, color='red', alpha=0.7)

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_perclass_ap50_test.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

# Deltas
print(f'\nPer-class AP50 improvement (YOLO11s - YOLO11n) on test:')
for i, cls in enumerate(CLASS_NAMES):
    delta = yolo11s_ap50[i] - yolo11n_ap50[i]
    flag = ' <- MINORITY' if cls in MINORITY_CLS else ''
    print(f'  {cls:<14}: {delta:+.1f}%{flag}')

---
## 4. Training Dynamics (Overlaid Curves)

In [ ]:
# ============================================================
# 4. TRAINING DYNAMICS — overlaid curves for both models
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training Dynamics — YOLO11n vs YOLO11s\n(100 epochs each, imgsz=1280)',
             fontsize=14, fontweight='bold')

plot_configs = [
    ('train/cls_loss',       axes[0, 0], 'Classification Loss'),
    ('train/box_loss',       axes[0, 1], 'Box Regression Loss'),
    ('train/dfl_loss',       axes[0, 2], 'DFL Loss'),
    ('metrics/mAP50(B)',     axes[1, 0], 'mAP50 (val)'),
    ('metrics/recall(B)',    axes[1, 1], 'Recall (val)'),
    ('metrics/precision(B)', axes[1, 2], 'Precision (val)'),
]

for col, ax, title in plot_configs:
    for df, label, color in [
        (yolo11n_results, 'YOLO11n', '#4C72B0'),
        (yolo11s_results, 'YOLO11s', '#DD8452'),
    ]:
        if col in df.columns:
            epochs = df.index + 1
            ax.plot(epochs, df[col], linewidth=2, color=color,
                    label=label, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.legend(fontsize=9)

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_training_dynamics.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

# Best epoch comparison
for label, df in [('YOLO11n', yolo11n_results), ('YOLO11s', yolo11s_results)]:
    best_ep = int(df['metrics/mAP50(B)'].idxmax()) + 1
    best_val = df['metrics/mAP50(B)'].max()
    print(f'{label}: best mAP50 = {best_val:.4f} @ epoch {best_ep}')

---
## 5. Minority Class Deep Dive (vest / no-vest)

These are the hardest classes — fewer training samples and visually similar to background clothing.
Tracking AP50 across checkpoints shows how quickly each model learns these classes.

In [ ]:
# ============================================================
# 5. MINORITY CLASS DEEP DIVE — checkpoint progression
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Minority Class AP50 Progression Across Checkpoints',
             fontsize=14, fontweight='bold')

for cls_idx, cls_name in enumerate(MINORITY_CLS):
    ax = axes[cls_idx]
    col = f'ap50_{cls_name}'

    for ckpt_df, label, color, marker in [
        (nb04_ckpt_df, 'YOLO11n', '#4C72B0', 'o'),
        (nb05_ckpt_df, 'YOLO11s', '#DD8452', 's'),
    ]:
        if col in ckpt_df.columns:
            x_labels = ckpt_df['checkpoint'].values
            y_vals = ckpt_df[col].values * 100
            ax.plot(range(len(x_labels)), y_vals, marker=marker,
                    linewidth=2, markersize=8, label=label, color=color)

    ax.set_title(f'{cls_name} AP50', fontweight='bold', fontsize=13)
    ax.set_ylabel('AP50 (%)')
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=45)
    ax.set_ylim(30, 100)
    ax.axhline(50, color='red', linestyle='--', alpha=0.5, label='Ablation threshold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_minority_progression.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

# Summary table
print('\nMinority class best.pt AP50 comparison:')
print(f'{"Class":<14} {"YOLO11n":>10} {"YOLO11s":>10} {"Delta":>10}')
print('-' * 46)
for cls in MINORITY_CLS:
    col = f'ap50_{cls}'
    n_val = nb04_ckpt_df[nb04_ckpt_df['checkpoint'] == 'best'][col].values[0] * 100
    s_val = nb05_ckpt_df[nb05_ckpt_df['checkpoint'] == 'best'][col].values[0] * 100
    print(f'{cls:<14} {n_val:>9.1f}% {s_val:>9.1f}% {s_val - n_val:>+9.1f}%')

---
## 6. Confusion Matrices (Side-by-Side)

In [ ]:
# ============================================================
# 6. CONFUSION MATRICES — side by side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('Normalized Confusion Matrices — Val Set',
             fontsize=14, fontweight='bold')

for ax, run_dir, label in [
    (axes[0], YOLO11N_DIR, 'YOLO11n'),
    (axes[1], YOLO11S_DIR, 'YOLO11s'),
]:
    cm_path = run_dir / 'confusion_matrix_normalized.png'
    if cm_path.exists():
        img = plt.imread(str(cm_path))
        ax.imshow(img)
        ax.set_title(label, fontsize=13, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'{label}\nConfusion matrix not found',
                ha='center', va='center', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_confusion_matrices.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

---
## 7. Radar Plot — Per-Class Performance Profile

In [ ]:
# ============================================================
# 7. RADAR PLOT — per-class AP50 on test set
# ============================================================
categories = CLASS_NAMES
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_title('Per-Class AP50 Profile (Test Set)', fontsize=14,
             fontweight='bold', pad=20)

for row_data, label, color in [
    (n_row, 'YOLO11n', '#4C72B0'),
    (s_row, 'YOLO11s', '#DD8452'),
]:
    values = [row_data[f'test_ap50_{cls}'] * 100 for cls in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(75, 100)
ax.set_yticks([80, 85, 90, 95, 100])
ax.set_yticklabels(['80', '85', '90', '95', '100'], fontsize=9)
ax.legend(loc='lower right', fontsize=11, bbox_to_anchor=(1.2, 0))
ax.grid(True)

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_radar_plot.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

---
## 8. Inference Speed Comparison

Benchmark both models on the val set to measure real-world inference latency.

In [ ]:
# ============================================================
# 8. INFERENCE SPEED BENCHMARK
# Run val on both models and capture speed metrics.
# ============================================================
import gc

speed_results = []

for label, run_dir, cfg_key in [
    ('YOLO11n', YOLO11N_DIR, 'yolo11n'),
    ('YOLO11s', YOLO11S_DIR, 'yolo11s'),
]:
    gc.collect()
    torch.cuda.empty_cache()

    weights = run_dir / 'weights' / 'best.pt'
    cfg = ALL_CONFIGS[cfg_key]
    model = YOLO(str(weights))

    # Warm-up + benchmark on val set
    m = model.val(
        data    = str(DATA_YAML),
        imgsz   = cfg['imgsz'],
        half    = True,
        device  = 0,
        split   = 'val',
        verbose = False,
    )

    # Speed is in ms per image: preprocess, inference, postprocess
    speed = m.speed  # dict with keys: preprocess, inference, postprocess
    total_ms = speed['preprocess'] + speed['inference'] + speed['postprocess']
    fps = 1000.0 / total_ms if total_ms > 0 else 0

    speed_results.append({
        'Model': label,
        'Preprocess (ms)': speed['preprocess'],
        'Inference (ms)': speed['inference'],
        'Postprocess (ms)': speed['postprocess'],
        'Total (ms)': total_ms,
        'FPS': fps,
        'Test mAP50': float(comp_df[comp_df['model'].str.contains(cfg_key)]['test_map50'].values[0]) * 100,
    })

    del model
    torch.cuda.empty_cache()
    print(f'{label}: {total_ms:.1f} ms/img ({fps:.1f} FPS)')

speed_df = pd.DataFrame(speed_results)
print(f'\nSpeed Comparison (GPU, FP16, imgsz=1280):')
print(speed_df.to_string(index=False, float_format='{:.1f}'.format))

In [ ]:
# ============================================================
# 8b. SPEED vs ACCURACY SCATTER PLOT
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title('Speed vs Accuracy Tradeoff (GPU, FP16, imgsz=1280)',
             fontsize=14, fontweight='bold')

colors = ['#4C72B0', '#DD8452']
for i, row in speed_df.iterrows():
    ax.scatter(row['Inference (ms)'], row['Test mAP50'],
              s=300, c=colors[i], zorder=5, edgecolors='black', linewidth=1.5)
    ax.annotate(row['Model'],
                xy=(row['Inference (ms)'], row['Test mAP50']),
                xytext=(12, 8), textcoords='offset points',
                fontsize=13, fontweight='bold', color=colors[i])

ax.set_xlabel('Inference Latency (ms/image)', fontsize=12)
ax.set_ylabel('Test mAP50 (%)', fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate the tradeoff
latency_delta = speed_df.iloc[1]['Inference (ms)'] - speed_df.iloc[0]['Inference (ms)']
map_delta = speed_df.iloc[1]['Test mAP50'] - speed_df.iloc[0]['Test mAP50']
ax.annotate('',
            xy=(speed_df.iloc[1]['Inference (ms)'], speed_df.iloc[1]['Test mAP50']),
            xytext=(speed_df.iloc[0]['Inference (ms)'], speed_df.iloc[0]['Test mAP50']),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--'))
mid_x = (speed_df.iloc[0]['Inference (ms)'] + speed_df.iloc[1]['Inference (ms)']) / 2
mid_y = (speed_df.iloc[0]['Test mAP50'] + speed_df.iloc[1]['Test mAP50']) / 2
ax.text(mid_x + 1, mid_y - 1,
        f'+{map_delta:.1f}% mAP50\n+{latency_delta:.1f}ms latency',
        fontsize=10, color='gray', style='italic')

plt.tight_layout()
plot_path = RESULTS_DIR / 'nb07_speed_vs_accuracy.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {plot_path}')

---
## 9. Final Summary & Deployment Recommendation

In [ ]:
# ============================================================
# 9. FINAL SUMMARY TABLE
# ============================================================
summary_data = []
for i, (label, cfg_key) in enumerate([('YOLO11n', 'yolo11n'), ('YOLO11s', 'yolo11s')]):
    row = comp_df[comp_df['model'].str.contains(cfg_key)].iloc[0]
    spd = speed_df[speed_df['Model'] == label].iloc[0]

    # Minority avg AP50 on test
    minority_test = (row['test_ap50_vest'] + row['test_ap50_no-vest']) / 2 * 100

    summary_data.append({
        'Model': label,
        'Params': model_info[i]['Parameters'],
        'Val mAP50': f"{row['val_map50']*100:.1f}%",
        'Test mAP50': f"{row['test_map50']*100:.1f}%",
        'Test mAP50-95': f"{row['test_map5095']*100:.1f}%",
        'Minority AP50': f"{minority_test:.1f}%",
        'Inference (ms)': f"{spd['Inference (ms)']:.1f}",
        'FPS': f"{spd['FPS']:.0f}",
    })

summary_df = pd.DataFrame(summary_data)
print('=' * 80)
print('  FINAL MODEL COMPARISON')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)

In [ ]:
# ============================================================
# 9b. DEPLOYMENT RECOMMENDATION
# ============================================================
print('=' * 70)
print('  DEPLOYMENT RECOMMENDATION')
print('=' * 70)
print()
print('Both models exceed 88% test mAP50 across all classes, including')
print('minority violation classes (vest, no-vest).')
print()
print('RECOMMENDED: YOLO11s for deployment')
print('  - Best overall accuracy: 93.2% test mAP50')
print('  - Strong minority class performance: >91% AP50 on vest/no-vest')
print('  - Real-time capable on GPU (see FPS above)')
print('  - Only ~3.6x the parameters of YOLO11n, but +4.7% test mAP50')
print()
print('ALTERNATIVE: YOLO11n for edge/CPU deployment')
print('  - Lighter model, faster inference')
print('  - Still strong at 88.6% test mAP50')
print('  - Better suited for resource-constrained environments')
print()
print('DEFERRED: RT-DETR-l')
print('  - Would provide CNN vs Transformer comparison')
print('  - Not justified given YOLO11s already exceeds 93% mAP50')
print('  - See NB06 for full rationale')
print()
print('=' * 70)
print('  Proceed to NB08 for Gradio deployment on Hugging Face Spaces')
print('=' * 70)

In [ ]:
# ============================================================
# 9c. SAVE FINAL SUMMARY TO RESULTS
# ============================================================
# Save summary as CSV for easy reference
summary_csv = RESULTS_DIR / 'nb07_final_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Summary saved -> {summary_csv}')

# Save speed benchmark
speed_csv = RESULTS_DIR / 'nb07_speed_benchmark.csv'
speed_df.to_csv(speed_csv, index=False)
print(f'Speed benchmark saved -> {speed_csv}')

# List all NB07 artifacts
print('\nAll NB07 artifacts:')
for f in sorted(RESULTS_DIR.glob('nb07_*')):
    size = f.stat().st_size
    unit = 'KB' if size > 1024 else 'B'
    size_display = size / 1024 if size > 1024 else size
    print(f'  {f.name:<40} {size_display:>8.1f} {unit}')

---
## Conclusion

### Key Findings

1. **YOLO11s is the clear winner** — +4.7% test mAP50 over YOLO11n with acceptable latency overhead
2. **Minority classes solved** — vest and no-vest both exceed 91% AP50 on the held-out test set (YOLO11s), up from ~85% with YOLO11n
3. **No ablation was needed** — both models passed the 50% minority AP50 gate comfortably
4. **Generalization is strong** — val-to-test gap is small (~1% mAP50), indicating no overfitting
5. **RT-DETR deferred** — diminishing returns at 93%+ mAP50 don't justify 40-50h compute

### Deployment Plan (NB08)

- **Primary model:** YOLO11s `best.pt` — Gradio app on Hugging Face Spaces
- **Fallback model:** YOLO11n `best.pt` — for CPU/edge scenarios
- Both models will be packaged with the Gradio interface for interactive PPE compliance detection